# BrainScaleS-2 primitive noise collection

This notebook only configures and invokes the canonical CLI. It does not duplicate collection or fitting logic. Use the `EBRAINS-experimental` kernel.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path('/mnt/user/shared/AnalogAttention')
SPIKING_CALIBRATION_PATH = None  # Path('/path/to/spiking_calibration.pbin')
HAGEN_CALIBRATION_PATH = None  # Path('/path/to/hagen_calibration.pbin')
RUN_PROBE = True
RUN_QUICK_SMOKE = False
RUN_OPERATING_POINT_SEARCH = False
RUN_FULL_COLLECTION = False
RUN_VALIDATE_EXISTING = False
EXISTING_OUTPUT_DIR = None
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_DIR = REPO_ROOT / 'artifacts' / 'brainscales2-primitives' / RUN_ID
CLI = REPO_ROOT / 'scripts' / 'evaluation' / 'brainscales2_primitive_noise.py'
assert CLI.is_file(), CLI

In [ ]:
def run_cli(*arguments):
    command = [sys.executable, str(CLI), *(str(value) for value in arguments)]
    print(' '.join(command), flush=True)
    subprocess.run(command, cwd=REPO_ROOT, check=True)

def calibration_args():
    if SPIKING_CALIBRATION_PATH is None or HAGEN_CALIBRATION_PATH is None:
        raise RuntimeError('Set explicit spiking and Hagen .pbin paths')
    return [
        '--spiking-calibration', Path(SPIKING_CALIBRATION_PATH).expanduser().resolve(),
        '--hagen-calibration', Path(HAGEN_CALIBRATION_PATH).expanduser().resolve(),
    ]

def spiking_calibration_args():
    if SPIKING_CALIBRATION_PATH is None:
        raise RuntimeError('Set an explicit spiking .pbin path')
    return [
        '--spiking-calibration', Path(SPIKING_CALIBRATION_PATH).expanduser().resolve(),
    ]

In [ ]:
if RUN_PROBE:
    run_cli('--phase', 'probe', '--backend', 'hardware', '--output-dir', OUTPUT_DIR)
    print((OUTPUT_DIR / 'probe.json').read_text())

In [ ]:
if RUN_QUICK_SMOKE:
    run_cli(
        '--phase', 'all', '--primitive', 'all', '--backend', 'hardware', '--quick',
        '--output-dir', OUTPUT_DIR / 'quick', *calibration_args(),
    )

In [ ]:
if RUN_OPERATING_POINT_SEARCH:
    search_root = OUTPUT_DIR / 'encoder_operating_point_search'
    search_common = [
        '--backend', 'hardware', '--repeats', 32, '--calibration-repeats', 16,
        '--device-count', 4, '--physical-coordinates', 3, 134, 273, 390,
        '--deadline', 900e-6, '--psi-ed-trial-guard', 2e-3,
        '--search-quick-codes',
        *spiking_calibration_args(),
    ]
    coarse_dir = search_root / 'coarse'
    run_cli(
        '--phase', 'optimize', '--primitive', 'phi-nl',
        '--search-current-stop-pairs',
        '1022:25', '512:45', '256:85', '128:165', '64:325', '32:645', '24:850',
        '--search-threshold-codes', 600, '--search-precharge-pairs', '1:63',
        '--search-max-candidates', 16, '--output-dir', coarse_dir, *search_common,
    )
    coarse = json.loads((coarse_dir / 'selected_operating_point.json').read_text())
    display(coarse['best_by_primitive'])
    refined = {}
    for primitive in ('phi-np', 'phi-nl'):
        candidate = coarse['best_by_primitive'][primitive]['candidate']
        current_stop = (
            f"{candidate['constant_current_code']}:"
            f"{candidate['ramp_stop_s'] * 1e6:.6g}"
        )
        refinement_dir = search_root / f'refine_{primitive}'
        run_cli(
            '--phase', 'optimize', '--primitive', primitive,
            '--search-current-stop-pairs', current_stop,
            '--search-threshold-codes', 500, 600, 700,
            '--search-precharge-pairs', '1:63', '2:32', '4:16',
            '--search-max-candidates', 16,
            '--output-dir', refinement_dir, *search_common,
        )
        refinement = json.loads(
            (refinement_dir / 'selected_operating_point.json').read_text()
        )
        refined[primitive] = refinement['best_by_primitive'][primitive]
        display(refinement['best_by_primitive'])
    for primitive, best in refined.items():
        candidate = best['candidate']
        current_stop = (
            f"{candidate['constant_current_code']}:"
            f"{candidate['ramp_stop_s'] * 1e6:.6g}"
        )
        precharge = (
            f"{candidate['precharge_input_fan_in']}:"
            f"{candidate['precharge_weight_maximum']}"
        )
        confirmation_dir = search_root / f'confirm_{primitive}'
        run_cli(
            '--phase', 'optimize', '--primitive', primitive, '--backend', 'hardware',
            '--repeats', 256, '--calibration-repeats', 128,
            '--device-count', 4, '--physical-coordinates', 3, 134, 273, 390,
            '--deadline', 900e-6, '--psi-ed-trial-guard', 2e-3,
            '--search-current-stop-pairs', current_stop,
            '--search-threshold-codes', candidate['threshold_code'],
            '--search-precharge-pairs', precharge,
            '--search-max-candidates', 1,
            '--output-dir', confirmation_dir, *spiking_calibration_args(),
        )
        display(json.loads(
            (confirmation_dir / 'selected_operating_point.json').read_text()
        )['best_by_primitive'])

In [ ]:
if RUN_FULL_COLLECTION:
    run_cli(
        '--phase', 'all', '--primitive', 'all', '--backend', 'hardware',
        '--output-dir', OUTPUT_DIR / 'full', *calibration_args(),
    )

In [ ]:
if RUN_VALIDATE_EXISTING:
    if EXISTING_OUTPUT_DIR is None:
        raise RuntimeError('Set EXISTING_OUTPUT_DIR')
    run_cli(
        '--phase', 'validate', '--backend', 'hardware',
        '--output-dir', Path(EXISTING_OUTPUT_DIR).expanduser().resolve(),
        *calibration_args(),
    )

result_dir = (OUTPUT_DIR / 'full') if (OUTPUT_DIR / 'full').is_dir() else OUTPUT_DIR / 'quick'
calibration = result_dir / 'primitive_noise_calibration.json'
if calibration.is_file():
    display(json.loads(calibration.read_text()))